# Stage 1 — Non-Instruction Fine-Tuning

**Domain:** Finance FAQ Assistant
**Base model:** `unsloth/Qwen2.5-0.5B` (loaded 4-bit via Unsloth)
**Goal:** adapt the base model to finance language, terminology, and writing
style by training on raw domain paragraphs (`data/non_instruction_data.txt`)
*before* any instruction tuning happens.

> Run this notebook on Google Colab with a T4 GPU: **Runtime → Change runtime type → T4 GPU**.


In [ ]:
# 1. Install dependencies (Colab)
%%capture
!pip install unsloth


In [ ]:
# Log in to Hugging Face Hub so we can push this stage's adapter for
# persistence (Colab local disk is wiped when the runtime disconnects).
from huggingface_hub import notebook_login
notebook_login()


In [ ]:
# 2. Get the project data onto the Colab filesystem.
# Option A: clone your GitHub repo (recommended once you've pushed it)
# !git clone https://github.com/<your-username>/finance-assistant-finetuning.git
# %cd finance-assistant-finetuning

# Option B: upload data/non_instruction_data.txt directly
from google.colab import files
import os

if not os.path.exists("data/non_instruction_data.txt"):
    os.makedirs("data", exist_ok=True)
    print("Upload non_instruction_data.txt now:")
    uploaded = files.upload()
    for fname in uploaded:
        os.rename(fname, f"data/{fname}")


In [ ]:
# 3. Load the base model with Unsloth (4-bit QLoRA)
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None          # auto-detect (bf16 on Colab T4-capable GPUs)
load_in_4bit = True   # QLoRA

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-0.5B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)


In [ ]:
# 4. Apply LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)


## 5. Load and chunk the raw domain text

`data/non_instruction_data.txt` contains 50+ raw finance paragraphs separated
by blank lines. For non-instruction (causal LM) fine-tuning we don't need any
instruction/response structure — we just want the model to keep reading and
predicting finance-domain text, so it absorbs vocabulary, tone, and recurring
concepts (interest rates, tax basis, liquidity, etc.).


In [ ]:
from datasets import Dataset

with open("data/non_instruction_data.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

paragraphs = [p.strip() for p in raw_text.split("\n\n") if p.strip()]
print(f"Loaded {len(paragraphs)} raw paragraphs")

# Chunk into ~max_seq_length-sized blocks so the packing collator has enough
# context per example, while keeping paragraph boundaries intact.
def chunk_paragraphs(paragraphs, max_chars=1500):
    chunks, current = [], ""
    for p in paragraphs:
        if len(current) + len(p) + 2 > max_chars and current:
            chunks.append(current.strip())
            current = ""
        current += p + "\n\n"
    if current.strip():
        chunks.append(current.strip())
    return chunks

chunks = chunk_paragraphs(paragraphs)
print(f"Built {len(chunks)} training chunks")

dataset = Dataset.from_dict({"text": chunks})


In [ ]:
# 6. Configure training (raw-text / non-instruction objective)
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        max_length = max_seq_length,
        packing = True,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,          # small demo run; raise for a fuller pass
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs/non_instruction",
        report_to = "none",
    ),
)


In [ ]:
# 7. Train
trainer_stats = trainer.train()


In [ ]:
HF_USERNAME = "Naveengangadhara"
STAGE1_REPO = f"{HF_USERNAME}/finance-qwen-stage1-adapter"

model.save_pretrained("outputs/non_instruction_adapter")
tokenizer.save_pretrained("outputs/non_instruction_adapter")
print("Saved Stage 1 adapter to outputs/non_instruction_adapter")

# Push to Hugging Face Hub so Stage 2 can load it even in a fresh runtime
model.push_to_hub(STAGE1_REPO)
tokenizer.push_to_hub(STAGE1_REPO)
print(f"Pushed Stage 1 adapter to https://huggingface.co/{STAGE1_REPO}")


## 9. Quick sanity test

Ask the Stage-1 model to continue a finance-flavored prompt. It should sound
more finance-native than the raw base model, even though it can't yet answer
questions well (that's Stage 2's job).


In [ ]:
FastLanguageModel.for_inference(model)

prompt = "When it comes to retirement savings, the key factors to consider are"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=120, use_cache=True)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))
